# Day 3: Pandas - GroupBy, Merge & Pivot

## Adding & Modifying Columns
Feature engineering — deriving new, more useful columns from existing ones — is one of the most common tasks in data preparation, whether you're computing a ratio, categorizing a value, or flagging a condition. `.apply()` runs a function (here, a `lambda`, a small anonymous function) on every value in a column and returns a new Series you can assign back as a new column. The example creates a `Grade` column by checking each `Score`: if it's 85 or above the student gets `'A'`, otherwise `'B'` — printing `df` afterward shows the original columns plus this new derived `Grade` column.

In [ ]:
import pandas as pd

df = pd.DataFrame({'Name': ['Alice', 'Bob', 'Charlie'], 'Score': [85, 92, 78]})
# apply() runs the lambda on every value in 'Score' to build the new 'Grade' column
df['Grade'] = df['Score'].apply(lambda x: 'A' if x >= 85 else 'B')
print(df)

## Handling Missing Data
Almost every real-world dataset has missing values — from sensor failures, unanswered survey questions, or failed joins — and how you handle them directly affects the accuracy of any analysis or model built on top. `isnull()` shows you where the gaps are (as a `True`/`False` grid), `fillna()` replaces missing values with something sensible (like 0 or a column average), and `dropna()` removes any row that contains at least one missing value entirely. In the output, `isnull()` marks the `NaN` positions as `True`, `fillna(0)` fills those same spots with 0, and `dropna()` keeps only the row that had no missing values at all (row 0).

In [ ]:
import numpy as np
df_nan = pd.DataFrame({'A': [1, np.nan, 3], 'B': [4, 5, np.nan]})
print(df_nan.isnull())   # True where a value is missing (NaN), False elsewhere
print(df_nan.fillna(0))  # replace every NaN with 0
print(df_nan.dropna())   # drop any row that contains at least one NaN

## GroupBy & Aggregation
GroupBy is the "split-apply-combine" pattern at the heart of most data analysis: split rows into groups based on a column's values, apply a calculation to each group separately, then combine the results — exactly what you'd do to answer "what's the average sales per city?" `groupby('City')['Sales'].mean()` groups rows by city and averages the `Sales` column within each group, while `.agg({'Sales': ['sum', 'max']})` computes multiple statistics (sum and max) for the same column at once. The first output shows one average sales value per city; the second shows a small table with both the total and the highest sale for each city.

In [ ]:
sales = pd.DataFrame({
    'City': ['NYC', 'NYC', 'LA', 'LA'],
    'Sales': [100, 150, 200, 130]
})
print(sales.groupby('City')['Sales'].mean())                  # average sales per city
print(sales.groupby('City').agg({'Sales': ['sum', 'max']}))   # multiple stats per city at once

## Merge & Concat
Real datasets are often split across multiple tables (e.g. customers in one file, orders in another), so combining them correctly is essential — this is the pandas equivalent of a SQL `JOIN`. `pd.merge()` combines two DataFrames based on a shared key column (`on='ID'`), and the `how` parameter controls what happens to rows that don't match: `'inner'` keeps only IDs present in both DataFrames, while `'left'` keeps every row from the left DataFrame and fills unmatched columns with `NaN`. In the output, the inner merge drops ID 3 (no score) and ID 4 (no name), while the left merge keeps ID 3 but shows `NaN` for its missing `Score`.

In [ ]:
df1 = pd.DataFrame({'ID': [1, 2, 3], 'Name': ['Alice', 'Bob', 'Charlie']})
df2 = pd.DataFrame({'ID': [1, 2, 4], 'Score': [85, 92, 78]})
print(pd.merge(df1, df2, on='ID', how='inner'))  # only IDs found in BOTH DataFrames
print(pd.merge(df1, df2, on='ID', how='left'))   # all IDs from df1; missing scores become NaN

## Pivot Table & Crosstab
Pivot tables reshape long-format data into a wide, spreadsheet-style summary — the same kind of "rows x columns" report you'd build with Excel's PivotTable feature — which makes it much easier to compare values across two categories at a glance. `pd.pivot_table()` lets you choose which column becomes the row index, which becomes the columns, and which aggregate function (`sum`, `mean`, etc.) combines the values that land in each cell; `pd.crosstab()` is a simpler shortcut specifically for counting how many rows fall into each combination of two categories. The pivot table output shows total sales for each Product/Region combination as a grid, while the crosstab shows how many records exist for each Product/Region pair.

In [ ]:
sales = pd.DataFrame({
    'Product': ['A', 'B', 'A', 'B'],
    'Region': ['East', 'West', 'East', 'West'],
    'Sales': [100, 150, 200, 130]
})
# Reshape into a Product x Region grid, summing Sales in each cell
print(pd.pivot_table(sales, values='Sales', index='Product', columns='Region', aggfunc='sum'))
# Count how many rows exist for each Product x Region combination
print(pd.crosstab(sales['Product'], sales['Region']))